# Substation Asset Health Monitoring — Metric Walkthrough

This notebook demonstrates all six health metric computations using the mock sensor data in `sample_data/sensor_readings.csv`.

**No credentials required.** This runs entirely on local mock data.

Metrics covered:
1. Mean Winding Temperature
2. Hotspot Temperature (IEEE C57.91 thermal model)
3. Thermal Aging Factor — FAA (IEC 60076-7 Arrhenius equation)
4. Overload Severity
5. Tap Changer Stress
6. Load-Temperature Sensitivity
7. Composite 0–100 Health Score + GREEN/AMBER/RED banding

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from health_metrics import (
    compute_mean_winding_temp,
    compute_hotspot_temperature,
    compute_faa,
    compute_overload_severity,
    compute_tap_changer_stress,
    compute_load_temp_sensitivity,
)
from composite_score import compute_composite_score, assign_risk_band, score_dataframe

print('Imports OK')

In [ ]:
# Load mock sensor data
df = pd.read_csv('../sample_data/sensor_readings.csv', parse_dates=['timestamp'])
master = pd.read_csv('../sample_data/transformers.csv')

# Merge nameplate MVA rating from master table
df = df.merge(master[['asset_id', 'mva_rated']], on='asset_id', how='left')

print(f'Loaded {len(df)} sensor rows for {df.asset_id.nunique()} assets')
df.head()

## Metric 1 — Mean Winding Temperature

Rolling average over a 24hr window. We use mean rather than peak (max) because
transient spikes during switching cause false alarms. A sustained elevated average
is a stronger signal of genuine thermal stress.

In [ ]:
results = {}

for asset_id, group in df.groupby('asset_id'):
    rolling_temp = compute_mean_winding_temp(group.copy(), window='2h', ts_col='timestamp')
    results[asset_id] = {'mean_winding_temp': rolling_temp.mean()}
    print(f'{asset_id}: mean winding temp = {rolling_temp.mean():.1f}°C (normal: 65–85°C)')

## Metric 2 + 3 — Hotspot Temperature and FAA

**Hotspot** is the hottest internal point (IEEE C57.91). Either measured directly
or derived from the thermal model: θ_H = θ_A + Δθ_TO + Δθ_H.

**FAA** (Thermal Aging Factor) quantifies insulation life consumption:

```
FAA = exp( 15000/383 − 15000/(273 + θ_H) )
FAA = 1.0  → aging at design rate
FAA = 4.0  → aging 4× faster — urgent attention required
```

In [ ]:
for asset_id, group in df.groupby('asset_id'):
    group = group.copy()
    # Add mock ambient temp (would come from IDW-weighted airport weather in production)
    group['ambient_temp_c'] = 8.0
    group['mva_rated_col'] = group['mva_rated']

    hotspot = compute_hotspot_temperature(
        group,
        ambient_col='ambient_temp_c',
        top_oil_col='oil_temp_c',
        winding_col='winding_temp_c',
        load_col='mva_actual',
        rating_col='mva_rated_col',
        hotspot_measured_col='hotspot_temp_c',
    )
    faa = compute_faa(hotspot)

    results[asset_id]['hotspot_temp']         = hotspot.mean()
    results[asset_id]['thermal_aging_factor'] = faa.mean()

    band = '🔴 CRITICAL' if faa.mean() > 4 else '🟡 ELEVATED' if faa.mean() > 2 else '🟢 NORMAL'
    print(f'{asset_id}: hotspot={hotspot.mean():.1f}°C  FAA={faa.mean():.2f}  {band}')

## Metrics 4–6 — Overload, Tap Changer, Load-Temp Sensitivity

In [ ]:
for asset_id, group in df.groupby('asset_id'):
    group = group.copy()
    group['mva_rated_col'] = group['mva_rated']

    overload = compute_overload_severity(group, load_col='mva_actual', rating_col='mva_rated_col',
                                         window='2h', ts_col='timestamp')
    tap = compute_tap_changer_stress(group, tposc_col='tposc', ts_col='timestamp', window='1h')
    sensitivity = compute_load_temp_sensitivity(group, load_col='mva_actual',
                                                temp_col='winding_temp_c', ts_col='timestamp', window=3)

    results[asset_id]['overload_severity']     = float(overload.sum())
    results[asset_id]['tap_changer_stress']    = float(tap.sum() if not tap.isna().all() else 0)
    results[asset_id]['load_temp_sensitivity'] = float(sensitivity.mean() if not sensitivity.isna().all() else 0)

    print(f'{asset_id}: overload={overload.sum():.2f}  tap_ops={tap.sum():.0f}  load_temp_r={sensitivity.mean():.2f}')

## Composite Score and Risk Banding

Weighted penalty sum → 0–100 health score → GREEN / AMBER / RED

Weights:
- Thermal Aging (FAA): 30% — irreversible damage, highest weight
- Hotspot Temperature: 25%
- Overload Severity:   20%
- Tap Changer Stress:  10%
- Mean Winding Temp:   10%
- Load-Temp Sensitivity: 5%

In [ ]:
summary_rows = []
for asset_id, metrics in results.items():
    score = compute_composite_score(metrics)
    band  = assign_risk_band(score)
    emoji = {'GREEN': '🟢', 'AMBER': '🟡', 'RED': '🔴'}[band]
    summary_rows.append({'asset_id': asset_id, 'health_score': score, 'risk_band': band, **metrics})
    print(f'{emoji} {asset_id}: score={score:.1f}  band={band}  FAA={metrics.get("thermal_aging_factor", 0):.2f}')

summary_df = pd.DataFrame(summary_rows).sort_values('health_score')
summary_df[['asset_id', 'health_score', 'risk_band', 'hotspot_temp', 'thermal_aging_factor']]

## Compare with pre-computed scores in health_scores.csv

The pipeline above uses 4 sensor rows per asset (1hr of data).
The CSV contains scores computed from the full production dataset.
Risk band direction should be consistent — RED assets stay RED.

In [ ]:
reference = pd.read_csv('../sample_data/health_scores.csv')
merged = summary_df.merge(reference[['asset_id','health_score','risk_band']], on='asset_id', suffixes=('_computed','_reference'))
merged[['asset_id','health_score_computed','risk_band_computed','health_score_reference','risk_band_reference']]